# 02 · Data Preparation

## Environment Setup and Pipeline Initialization

In [1]:
# External library dependencies for data manipulation and interactive notebook rendering
import pandas as pd
from IPython.display import display

# Custom domain-specific module importing configuration constants, ETL routines, and splitting functions
from data_pipeline import (
    # Pipeline configuration and dataset metadata constants
    DATASETS,
    EXPERIMENT,
    # Data cleaning and preprocessing functions
    apply_cleaning,
    # Integrity validation and feature audit utilities
    check_subsets,
    feature_overlap_report,
    # Ingestion and serialization state-management routines
    load_prepared_experiment,
    load_raw,
    save_experiment,
    # Cryptographic hash utility for data provenance and cache validation
    sha256_file,
    # Feature-target separation, balancing, and class-stratified partitioning utilities
    make_balanced_subsets,
    split_features_target,
    stratified_split_indices,
)

# Unset maximum column rendering limit to prevent DataFrame truncation during interactive inspection
pd.set_option("display.max_columns", None)

## Load and validate raw data

In [2]:
# Initialize dictionaries to hold loaded DataFrames and their SHA-256 signatures
raw, raw_hashes = {}, {}

# Ingest datasets, compute provenance hashes, and validate missing-value constraints
for name, cfg in DATASETS.items():
    raw[name] = load_raw(cfg)
    raw_hashes[name] = sha256_file(cfg.path)

    # Fail-fast check: compute total missing cells across all columns to enforce complete data assumption
    n_missing = int(raw[name].isna().sum().sum())
    if n_missing:
        raise ValueError(f"{name}: {n_missing} missing values found; no imputation step exists.")

# Aggregate metadata into a transposed DataFrame for tabular rendering in the notebook
display(
    pd.DataFrame(
        {
            name: {
                "rows": len(df),
                "columns": df.shape[1],
                "target": DATASETS[name].target,
                # Extract and sort unique class labels from the dataset's target column
                "target classes": sorted(df[DATASETS[name].target].unique().tolist()),
                # Truncate cryptographic signature to the first 12 characters for display brevity
                "raw sha256 (first 12)": raw_hashes[name][:12],
            }
            for name, df in raw.items()
        }
    ).T  # Transpose to orient individual datasets as rows and attributes as columns
)

,rows,columns,target,target classes,raw sha256 (first 12)
diabetes,253680,22,Diabetes_binary,"[0.0, 1.0]",19f367e3e335
cardiovascular,70000,13,cardio,"[0, 1]",21a705d23381


## Clean

`apply_cleaning` runs, in order:

1. **Exact-duplicate removal** on the modelling columns (identifiers ignored), keeping the first occurrence. Prevents identical rows from ending up in both train and test.
2. **Validity rules** (cardiovascular): physiological ranges for blood pressure, height and weight, plus systolic > diastolic.
3. **Transforms:** `age` days → `age_years` (÷ 365.25); `gender` 1/2 → 0/1.
4. **Compact dtypes:** integer-valued columns → `int8`/`int16`, checked so no value can be truncated.

The log shows how each step changes the row count and the positive-class prevalence.

In [3]:
# Container dictionary mapping dataset identifiers to their respective cleaning result objects
cleaning = {}

# Execute pipeline cleaning transformations and log diagnostic output per dataset
for name, cfg in DATASETS.items():
    # Apply dataset-specific cleaning transformation pipeline configured in `cfg`
    cleaning[name] = apply_cleaning(raw[name], cfg)
    result = cleaning[name]

    print(f"\n{name.upper()}")

    # Render step-by-step transformation audit log, setting the step name as index for readability
    display(result.log.set_index("step").round(4))

    # Output domain rule violation counts if post-deduplication quality constraints were breached
    if not result.rule_violations.empty:
        display(result.rule_violations.rename("rows violating (measured after de-duplication)").to_frame())

    # Calculate deep memory footprint reduction (bytes to MB) accounting for object dtype pointers
    print(
        f"Memory: {raw[name].memory_usage(deep=True).sum() / 1e6:.1f} MB -> "
        f"{result.data.memory_usage(deep=True).sum() / 1e6:.1f} MB"
    )


DIABETES


,rows_before,rows_removed,rows_after,pct_removed,target_prevalence
step,,,,,
raw,253680,0,253680,0.00,0.1393
1. drop exact duplicates,253680,24206,229474,9.54,0.1529
2. apply validity rules,229474,0,229474,0.00,0.1529


Memory: 44.6 MB -> 7.6 MB

CARDIOVASCULAR


,rows_before,rows_removed,rows_after,pct_removed,target_prevalence
step,,,,,
raw,70000,0,70000,0.00,0.4997
1. drop exact duplicates,70000,24,69976,0.03,0.4998
2. apply validity rules,69976,1364,68612,1.95,0.4948


,rows violating (measured after de-duplication)
"ap_hi outside [60, 250]",228
"ap_lo outside [40, 160]",1025
"height outside [100, 220]",30
"weight outside [30, 200]",7
ap_hi <= ap_lo,1236


Memory: 7.3 MB -> 3.1 MB


## Feature matrix and target vector

In [5]:
# Dictionary storing separated predictor matrices (X) and target vectors (y) per dataset
prepared = {}

# Partition datasets, validate structural schema compliance, and analyze target distributions
for name, cfg in DATASETS.items():
    # Decouple cleaned DataFrame into design matrix X (features) and response array y (target)
    X, y = split_features_target(cleaning[name].data, cfg)

    # Flatten and sort expected column names across all feature categories in the configuration
    expected = sorted(col for cols in cfg.cleaned_feature_types().values() for col in cols)

    # Schema integrity check: enforce strict alignment between extracted and expected features
    if sorted(X.columns) != expected:
        raise ValueError(f"{name}: feature columns do not match the configuration.")

    # Store partitioned components and display output matrix dimensions
    prepared[name] = {"X": X, "y": y}
    print(f"{name.upper()}: X {X.shape}, y {y.shape}")

    # Render target class frequency table sorted by class index to evaluate label imbalance
    display(y.value_counts().sort_index().rename("count").to_frame())

DIABETES: X (229474, 21), y (229474,)


,count
Diabetes_binary,
0,194377
1,35097


CARDIOVASCULAR: X (68612, 11), y (68612,)


,count
cardio,
0,34664
1,33948


## Experimental configuration

In [6]:
print(f"Test size:          {EXPERIMENT.test_size:.0%}")
print(f"Test split seed:    {EXPERIMENT.test_seed}")
print(f"Training seeds:     {list(EXPERIMENT.training_seeds)}")
print(f"Training sizes:     {list(EXPERIMENT.training_sizes)}")

Test size:          20%
Test split seed:    42
Training seeds:     [42, 123, 456, 789, 1011]
Training sizes:     [8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]


## Stratified train/test split

The split works on index labels (raw row ids), so every row stays traceable. The external test set keeps the dataset's natural class prevalence and is never resampled.

In [7]:
# Containers to store raw partition index tuples and the resulting split DataFrames per dataset
splits, split_data = {}, {}

# Partition each dataset into stratified train and test subsets
for name, data in prepared.items():
    X, y = data["X"], data["y"]

    # Compute label-preserving train/test split indices using target distribution 'y' and global 'EXPERIMENT' parameters
    train_idx, test_idx = stratified_split_indices(y, EXPERIMENT)
    splits[name] = (train_idx, test_idx)

    # Slice feature matrices and target vectors using label-based explicit index selection (.loc)
    split_data[name] = {
        "X_train": X.loc[train_idx],
        "y_train": y.loc[train_idx],
        "X_test": X.loc[test_idx],
        "y_test": y.loc[test_idx],
    }

# Render a summary table verifying sample counts and target positive rates across partitions
display(
    pd.DataFrame.from_dict(
        {
            name: {
                "training pool": len(d["y_train"]),
                "test set": len(d["y_test"]),
                # Calculate the positive target proportion (mean of binary labels) to confirm stratification fidelity
                "train positive rate": d["y_train"].mean(),
                "test positive rate": d["y_test"].mean(),
            }
            for name, d in split_data.items()
        },
        orient="index",  # Orient keys (dataset names) as rows and metric attributes as columns
    ).round(4)
)

,training pool,test set,train positive rate,test positive rate
diabetes,183579,45895,0.1529,0.1529
cardiovascular,54889,13723,0.4948,0.4948


## Feasibility: can the minority class support the largest balanced subset?

In [8]:
for name, d in split_data.items():
    counts = d["y_train"].value_counts()
    needed = max(EXPERIMENT.training_sizes) // len(counts)
    if counts.min() < needed:
        raise ValueError(
            f"{name}: minority class has {counts.min():,} training rows, {needed:,} are needed."
        )
    print(f"{name.upper()}: minority class has {counts.min():,} rows (need {needed:,}) - OK")

DIABETES: minority class has 28,078 rows (need 4,096) - OK
CARDIOVASCULAR: minority class has 27,158 rows (need 4,096) - OK


## Class-balanced subsampling

In [9]:
subsets = {
    name: make_balanced_subsets(d["y_train"], EXPERIMENT.training_sizes, EXPERIMENT.training_seeds)
    for name, d in split_data.items()
}
for name, by_size in subsets.items():
    print(f"{name.upper()}: {len(by_size)} training sizes x {len(EXPERIMENT.training_seeds)} seeds")

DIABETES: 11 training sizes x 5 seeds
CARDIOVASCULAR: 11 training sizes x 5 seeds


## Subsample integrity checks

For every subset: correct size, no repeated rows, all rows inside the training pool, none in the test set, exact class balance, no missing values, correct feature count.

In [10]:
for name, d in split_data.items():
    n_checked = check_subsets(subsets[name], d["X_train"], d["y_train"], splits[name][1])
    print(f"{name.upper()}: {n_checked} subsets passed all integrity checks")

DIABETES: 55 subsets passed all integrity checks
CARDIOVASCULAR: 55 subsets passed all integrity checks


## Leakage and overlap audit

In [11]:
# Assert partition disjointness to prevent data leakage between training and testing sets
for name, (train_idx, test_idx) in splits.items():
    # Fail-fast check: ensure no sample index exists simultaneously in both train and test splits
    if train_idx.isin(test_idx).any():
        raise AssertionError(f"{name}: train and test indices overlap.")

# Compute feature-space duplicate overlap metrics between train and test sets per dataset
overlap = pd.DataFrame(
    {
        name: feature_overlap_report(d["X_train"], d["y_train"], d["X_test"], d["y_test"])
        for name, d in split_data.items()
    }
).T  # Transpose to align dataset identifiers as rows and audit metrics as columns

# Cast frequency and count metrics from float to integer data types for display consistency
overlap["test_rows"] = overlap["test_rows"].astype(int)
overlap["test_rows_with_identical_train_profile"] = overlap["test_rows_with_identical_train_profile"].astype(int)

print("Train/test row overlap: none.")
# Display the compiled data leakage audit table rounded to 4 decimal places
display(overlap.round(4))

Train/test row overlap: none.


,test_rows,test_rows_with_identical_train_profile,share_with_identical_train_profile,label_agreement
diabetes,45895,502,0.0109,0.0
cardiovascular,13723,8,0.0006,0.0


## Determinism check

Regenerating all subsets from the same seeds must give identical index lists.

In [12]:
for name, d in split_data.items():
    again = make_balanced_subsets(d["y_train"], EXPERIMENT.training_sizes, EXPERIMENT.training_seeds)
    if again != subsets[name]:
        raise AssertionError(f"{name}: subsampling is not reproducible.")
    print(f"{name.upper()}: subsampling is deterministic")

DIABETES: subsampling is deterministic
CARDIOVASCULAR: subsampling is deterministic


## Summary

In [14]:
# Aggregate metadata across pipeline stages (ingestion, cleaning, splitting) into a unified summary DataFrame
summary = pd.DataFrame.from_dict(
    {
        name: {
            # Instance count prior to cleaning and filtering
            "raw rows": len(raw[name]),
            # Post-cleaning observation count and feature space dimensionality
            "cleaned rows": len(prepared[name]["X"]),
            "features": prepared[name]["X"].shape[1],
            # Partition sizes across stratified training pool and holdout testing sets
            "training pool": len(split_data[name]["y_train"]),
            "test rows": len(split_data[name]["y_test"]),
            # Positive class ratio in test set, defining the uninformative baseline (random chance level) for PR-AUC
            "test prevalence (PR-AUC chance level)": round(split_data[name]["y_test"].mean(), 4),
            # Target positive class proportion enforced during downstream balanced sub-sampling
            "training-subset prevalence": 0.5,
        }
        for name in DATASETS
    },
    orient="index",  # Set dataset identifiers as DataFrame index rows and metadata attributes as columns
)

# Render consolidated experimental overview table
display(summary)

,raw rows,cleaned rows,features,training pool,test rows,test prevalence (PR-AUC chance level),training-subset prevalence
diabetes,253680,229474,21,183579,45895,0.1529,0.5
cardiovascular,70000,68612,11,54889,13723,0.4948,0.5


## Save

Written to `data/processed/`: one `cleaned.csv.gz` per dataset plus `experiment_manifest.json` (config, raw hashes, package versions, cleaning log, split and subsample indices). Subsamples are stored as index lists, so nothing is duplicated on disk.

In [15]:
manifest_path = save_experiment(
    cleaned={name: result.data for name, result in cleaning.items()},
    splits=splits,
    subsets=subsets,
    cleaning_logs={name: result.log for name, result in cleaning.items()},
    raw_hashes=raw_hashes,
    exp=EXPERIMENT,
)
print(f"Manifest: {manifest_path}")
print(f"Size:     {manifest_path.stat().st_size / 1e6:.2f} MB")

Manifest: C:\Users\ASUS\IU Thesis\data\processed\experiment_manifest.json
Size:     3.38 MB


## Verify the round-trip

Reloads from disk and compares against the in-memory objects (values, dtypes, indices, subsets), not just the keys.

Downstream notebooks load the experiment with:

```python
from data_pipeline import load_prepared_experiment
experiment = load_prepared_experiment()
subset = experiment["training_subsets"]["diabetes"][128][42]   # size 128, seed 42
X_sub, y_sub = subset["X"], subset["y"]
X_test, y_test = experiment["split_data"]["diabetes"]["X_test"], experiment["split_data"]["diabetes"]["y_test"]
```

In [16]:
loaded = load_prepared_experiment()

for name in DATASETS:
    pd.testing.assert_frame_equal(loaded["prepared_data"][name]["X"], prepared[name]["X"])
    pd.testing.assert_series_equal(loaded["prepared_data"][name]["y"], prepared[name]["y"])

    for part in ("X_train", "X_test"):
        pd.testing.assert_frame_equal(loaded["split_data"][name][part], split_data[name][part])
    for part in ("y_train", "y_test"):
        pd.testing.assert_series_equal(loaded["split_data"][name][part], split_data[name][part])

    train_idx, test_idx = splits[name]
    assert list(loaded["split_data"][name]["train_indices"]) == list(train_idx)
    assert list(loaded["split_data"][name]["test_indices"]) == list(test_idx)

    for size, by_seed in subsets[name].items():
        for seed, idx in by_seed.items():
            assert list(loaded["training_subsets"][name][size][seed]["indices"]) == idx

print("Round-trip verified: values, dtypes, indices and all subsets are identical.")
print("Saved config:", loaded["config"])

Round-trip verified: values, dtypes, indices and all subsets are identical.
Saved config: {'test_size': 0.2, 'test_seed': 42, 'training_seeds': [42, 123, 456, 789, 1011], 'training_sizes': [8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192], 'n_classes': 2}
